# 🦜🔍 LangSmith — Complete  Guide
### Tracing · Prompt Version Control · Golden Datasets · Evaluation
---
> **Models used:** `gpt-4o-mini` (chat) · `text-embedding-3-small` (embeddings)  
> **Packages:** `langsmith==0.8.x` · `langchain==1.3.x` · `langchain-openai==1.2.x`  
> **Secrets loaded from:** Google Colab Secret Manager (🔑 left panel)

---

## 📚 Table of Contents
1. [Setup & Installation](#setup)
2. [Configure Secrets (Colab)](#secrets)
3. [Module 1 — LangSmith Tracing Fundamentals](#tracing)
4. [Module 2 — @traceable Decorator & Custom Spans](#traceable)
5. [Module 3 — Prompt Version Control (Hub)](#prompt-versioning)
6. [Module 4 — Golden Datasets](#golden-datasets)
7. [Module 5 — Evaluation with evaluate()](#evaluation)
8. [Module 6 — Full RAG Pipeline with All Features](#rag-pipeline)
9. [Module 7 — Feedback & Human Annotation](#feedback)


---
## 1. 🛠️ Setup & Installation <a name="setup"></a>
Install all required packages. This cell only needs to run once per Colab session.


In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────────
!pip install -q \
    langsmith>=0.8.0 \
    langchain>=1.3.0 \
    langchain-openai>=1.2.0 \
    langchain-core>=0.3.0 \
    langchain-text-splitters>=0.3.0 \
    langchain-community>=0.4.0 \
    langchain-chroma>=0.2.0 \
    chromadb \
    faiss-cpu \
    tiktoken \
    openai

print("✅ All packages installed")


✅ All packages installed


---
## 2. 🔑 Configure Secrets (Colab Secret Manager) <a name="secrets"></a>

**Before running this cell:**
1. Click the 🔑 **key icon** in the left sidebar
2. Add the following secrets:
   - `OPENAI_API_KEY` — your OpenAI key
   - `LANGCHAIN_API_KEY` — your LangSmith API key (from smith.langchain.com → Settings → API Keys)
   - `LANGCHAIN_PROJECT` — a project name like `llmops-day14` (will be created if it doesn't exist)

> **Note:** `LANGCHAIN_TRACING_V2=true` enables automatic tracing for all LangChain chains.


In [ ]:
import os
from google.colab import userdata

# ── Load secrets from Colab Secret Manager ────────────────────────────────────
os.environ["OPENAI_API_KEY"]        = userdata.get("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"]     = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]  = "true"          # Enable auto-tracing
os.environ["LANGCHAIN_PROJECT"]     = "llmops-day14"
os.environ["LANGCHAIN_ENDPOINT"]    = "https://api.smith.langchain.com"

# ── Verify connection ──────────────────────────────────────────────────────────
from langsmith import Client
client = Client()

# Test connection
projects = list(client.list_projects())
print(f"✅ Connected to LangSmith")
print(f"📂 Available projects: {[p.name for p in projects[:5]]}")
print(f"🎯 Active project: {os.environ['LANGCHAIN_PROJECT']}")


✅ Connected to LangSmith
📂 Available projects: ['rag-pipeline-v1-96a03715', 'support-bot-v1-comparison-d63e7707', 'evaluators', 'support-bot-v3-defe7107', 'llmops-day14']
🎯 Active project: llmops-day14


---
## 3. 🔭 Module 1 — LangSmith Tracing Fundamentals <a name="tracing"></a>

### What Is Tracing?
LangSmith tracing records every step of your LLM pipeline:
- **Runs** = individual LLM calls, chain steps, tool calls
- **Traces** = a full tree of runs triggered by one user request
- **Spans** = named sections of work (auto-created for LangChain components)

### How It Works
Setting `LANGCHAIN_TRACING_V2=true` **automatically instruments** all LangChain components — no code changes needed. Every `chain.invoke()`, `llm.invoke()`, and `retriever.get_relevant_documents()` generates a trace.


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Models ─────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    # LangSmith tracing is automatic — no extra config needed here
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536  # Default; can reduce to 512 or 256 for efficiency
)

print(f"✅ LLM: {llm.model_name}")
print(f"✅ Embeddings: {embeddings.model}")


✅ LLM: gpt-4o-mini
✅ Embeddings: text-embedding-3-small


In [ ]:
# ── Simple traced chain ────────────────────────────────────────────────────────
# This chain is AUTOMATICALLY traced because LANGCHAIN_TRACING_V2=true

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an LLMOps expert. Be concise and practical."),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

# Every invocation creates a trace in LangSmith
response = chain.invoke({
    "question": "What is the difference between LangSmith and Langfuse?"
})

print("Response:", response)
print()
print("🔍 View this trace at: https://smith.langchain.com")
print(f"📁 Project: {os.environ['LANGCHAIN_PROJECT']}")


Response: LangSmith and Langfuse are both tools designed to enhance the development and management of language models, but they serve different purposes:

1. **LangSmith**: Primarily focuses on providing a framework for building and testing language models. It emphasizes the development lifecycle, allowing users to create, iterate, and refine models with features like version control and testing environments.

2. **Langfuse**: Concentrates on monitoring and analyzing the performance of deployed language models. It offers insights into user interactions, model behavior, and performance metrics, helping teams optimize and troubleshoot their models in real-time.

In summary, LangSmith is more about model development, while Langfuse is focused on model monitoring and analytics.

🔍 View this trace at: https://smith.langchain.com
📁 Project: llmops-day14


In [ ]:
# ── Adding metadata to traces ──────────────────────────────────────────────────
# Use RunnableConfig to attach metadata, tags, and run names to traces

response2 = chain.invoke(
    {"question": "Explain semantic caching in 2 sentences"},
    config={
        "run_name": "semantic-caching-explanation",   # Human-readable name in UI
        "tags": ["day-14", "llmops", "caching"],       # Filter by these in UI
        "metadata": {                                   # Custom key-value pairs
            "student_cohort": "batch-7",
            "module": "llmops",
            "difficulty": "intermediate",
        }
    }
)

print("Response:", response2)
print()
print("✅ Trace created with metadata tags — filter by 'day-14' in LangSmith UI")


Response: Semantic caching is a technique that stores the results of previously executed queries along with their semantic context, allowing for faster retrieval of relevant data in future queries. By leveraging the relationships and meanings of the data, it reduces the need for redundant computations and improves overall query performance.

✅ Trace created with metadata tags — filter by 'day-14' in LangSmith UI


In [ ]:
# ── Batch tracing — run multiple inputs and see all traces ────────────────────
questions = [
    "What is vLLM used for?",
    "Explain the KV cache in one sentence",
    "What is the difference between Canary and Blue-Green deployment?",
    "When would you use DVC over MLflow?",
]

print("Running batch of questions...")
results = chain.batch(
    [{"question": q} for q in questions],
    config={"tags": ["batch-demo", "day-14"]}
)

for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r[:100]}...")
    print()

print("✅ All 4 traces visible in LangSmith under project:", os.environ["LANGCHAIN_PROJECT"])


Running batch of questions...
Q: What is vLLM used for?
A: vLLM is a high-performance library designed for efficient inference of large language models. It opt...

Q: Explain the KV cache in one sentence
A: The KV cache (Key-Value cache) stores the key-value pairs of past computations in language models to...

Q: What is the difference between Canary and Blue-Green deployment?
A: Canary and Blue-Green deployments are both strategies for releasing new software versions, but they ...

Q: When would you use DVC over MLflow?
A: Use DVC (Data Version Control) over MLflow when:

1. **Data Management**: You need robust data versi...

✅ All 4 traces visible in LangSmith under project: llmops-day14


---
## 4. 🎯 Module 2 — @traceable Decorator & Custom Spans <a name="traceable"></a>

### Why Use @traceable?
`@traceable` lets you trace **any Python function** — not just LangChain components.
Use it to trace:
- Pre/post-processing steps
- Custom retrieval logic
- Business logic around your LLM calls
- Any function you want to see in the trace tree


In [ ]:
from langsmith import traceable, Client
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import time

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ── @traceable on a simple function ───────────────────────────────────────────
@traceable(name="preprocess-user-query", tags=["preprocessing"])
def preprocess_query(raw_query: str) -> str:
    """Cleans and normalises the user query before sending to LLM."""
    cleaned = raw_query.strip().lower()
    # Remove filler words
    for word in ["please", "can you", "could you", "tell me"]:
        cleaned = cleaned.replace(word, "")
    return cleaned.strip()

# ── @traceable on a retrieval function ────────────────────────────────────────
@traceable(name="custom-retrieval", run_type="retriever")
def retrieve_context(query: str, documents: list[dict]) -> list[str]:
    """Simple keyword-based retrieval — traced as a 'retriever' span."""
    query_words = set(query.lower().split())
    scored = []
    for doc in documents:
        content_words = set(doc["content"].lower().split())
        score = len(query_words & content_words) / len(query_words)
        if score > 0:
            scored.append((score, doc["content"]))

    scored.sort(reverse=True)
    return [content for _, content in scored[:3]]

# ── @traceable on the full pipeline ───────────────────────────────────────────
@traceable(name="full-qa-pipeline", tags=["pipeline", "day-14"])
def run_qa_pipeline(user_query: str, knowledge_base: list[dict]) -> dict:
    """Full QA pipeline — each step appears as a child span in LangSmith."""
    start = time.time()

    # Step 1: Preprocess (traced as child span)
    clean_query = preprocess_query(user_query)

    # Step 2: Retrieve (traced as child span)
    context_docs = retrieve_context(clean_query, knowledge_base)
    context_str   = "\n".join(context_docs)

    # Step 3: Generate answer with LangChain (auto-traced)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer based only on the provided context. Be concise.\n\nContext:\n{context}"),
        ("human", "{question}")
    ])
    chain  = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context_str, "question": clean_query})

    return {
        "query": user_query,
        "cleaned_query": clean_query,
        "num_docs_retrieved": len(context_docs),
        "answer": answer,
        "latency_ms": round((time.time() - start) * 1000, 2)
    }

# ── Sample knowledge base ──────────────────────────────────────────────────────
KB = [
    {"id": 1, "content": "vLLM is a high-throughput inference engine using PagedAttention for memory management."},
    {"id": 2, "content": "LangSmith provides tracing, datasets, and evaluation for LLM applications."},
    {"id": 3, "content": "Semantic caching stores LLM responses by meaning, not exact text, reducing API calls."},
    {"id": 4, "content": "MLflow tracks LLM experiments: prompts, parameters, metrics and artifacts."},
    {"id": 5, "content": "Canary releases gradually route traffic to new model versions to reduce risk."},
    {"id": 6, "content": "The KV cache in transformers stores key-value pairs to avoid recomputation."},
]

result = run_qa_pipeline("What tools can I use for LLM tracing?", KB)
print("Answer:", result["answer"])
print("Latency:", result["latency_ms"], "ms")
print("\n✅ Check LangSmith — you'll see a trace tree with 3 child spans")


Answer: You can use LangSmith for LLM tracing.
Latency: 694.29 ms

✅ Check LangSmith — you'll see a trace tree with 3 child spans


In [ ]:
# ── @traceable with metadata from run context ──────────────────────────────────

from langsmith.run_helpers import traceable, get_current_run_tree

@traceable(name="contextual-generation", run_type="llm")
def generate_with_context(prompt_text: str, user_id: str, feature: str) -> str:

    run_tree = get_current_run_tree()

    if run_tree:
        # Update existing metadata instead of assigning
        run_tree.metadata.update({
            "user_id": user_id,
            "feature": feature,
            "model": "gpt-4o-mini"
        })

    response = llm.invoke(prompt_text)
    return response.content


result = generate_with_context(
    "List 3 key benefits of using LangSmith in production",
    user_id="student-001",
    feature="llmops-module"
)

print(result)
print("\n✅ user_id and feature metadata attached to this trace")

LangSmith is a tool designed to enhance the development and deployment of language models. Here are three key benefits of using LangSmith in production:

1. **Streamlined Model Management**: LangSmith provides a centralized platform for managing multiple language models, allowing teams to easily track versions, monitor performance, and manage updates. This streamlining helps reduce the complexity associated with deploying and maintaining various models in production.

2. **Enhanced Collaboration**: With features that facilitate collaboration among team members, LangSmith enables developers, data scientists, and stakeholders to work together more effectively. This can lead to faster iteration cycles, improved feedback loops, and ultimately a more refined product.

3. **Robust Monitoring and Analytics**: LangSmith offers tools for monitoring model performance in real-time, providing insights into usage patterns, accuracy, and potential issues. This data-driven approach allows teams to ma

---
## 5. 📝 Module 3 — Prompt Version Control (LangSmith Hub) <a name="prompt-versioning"></a>

### Why Version Prompts?
Prompts are the core of LLM behavior. Versioning them like code gives you:
- **Rollback** — instantly revert to a previous working prompt
- **History** — know exactly what changed between versions
- **Audit trail** — who changed what, when, and why
- **Reuse** — share prompts across projects and teams
- **A/B testing** — run two prompt versions simultaneously

### The LangSmith Prompt Hub
Prompts are stored in LangSmith with full version history (git-like commits).
Each version has a hash, tags, and description.


In [ ]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

client = Client()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Create and push a prompt (version 1)
# ─────────────────────────────────────────────────────────────────────────────
PROMPT_NAME = "support-bot-prompt"   # Use a unique name for your workspace

# Version 1 — basic prompt
prompt_v1 = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful customer support agent."),
    ("human", "{user_message}")
])

# Push to LangSmith Hub
v1_commit = client.push_prompt(
    PROMPT_NAME,
    object=prompt_v1,
    is_public=False,
    description="Initial version of support bot prompt",
    tags=["v1", "baseline"],
    commit_description="Initial prompt: basic support agent"
)

print(f"✅ Pushed prompt v1")
print(f"   Commit hash: {v1_commit}")
print(f"   View at: https://smith.langchain.com/hub/{PROMPT_NAME}")


✅ Pushed prompt v1
   Commit hash: https://smith.langchain.com/prompts/support-bot-prompt/a1b4ef45?organizationId=302c62e7-0cd9-4a40-ab47-5a2a9cc38523
   View at: https://smith.langchain.com/hub/support-bot-prompt


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Improve the prompt and push version 2
# ─────────────────────────────────────────────────────────────────────────────

# Version 2 — more detailed with persona and constraints
prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful, empathetic customer support agent for an e-commerce company.

Guidelines:
- Always acknowledge the customer's issue before providing a solution
- Keep responses under 150 words unless the issue is complex
- If you cannot resolve an issue, escalate with: "I'll connect you with a specialist"
- Always end with asking if there's anything else you can help with
- Tone: professional but warm"""),
    ("human", "{user_message}")
])

v2_commit = client.push_prompt(
    PROMPT_NAME,
    object=prompt_v2,
    is_public=False,
    description="Added persona, guidelines, tone instructions",
    tags=["v2", "improved"],
    commit_description="v2: Added persona, constraints, escalation path"
)

print(f"✅ Pushed prompt v2")
print(f"   Commit hash: {v2_commit}")


✅ Pushed prompt v2
   Commit hash: https://smith.langchain.com/prompts/support-bot-prompt/2c06391c?organizationId=302c62e7-0cd9-4a40-ab47-5a2a9cc38523


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Push a version 3 with few-shot examples
# ─────────────────────────────────────────────────────────────────────────────

prompt_v3 = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful, empathetic customer support agent for an e-commerce company.

Guidelines:
- Acknowledge the issue, then solve it
- Responses under 150 words
- Escalate complex issues: "I'll connect you with a specialist"
- End by asking if there's more you can help with"""),
    ("human", "My order hasn't arrived in 2 weeks"),
    ("assistant", """I'm sorry to hear your order hasn't arrived yet! That must be frustrating.

Please share your order number and I'll look into this right away. Typically, I can:
1. Check the current shipping status
2. Contact the carrier for an update
3. Arrange a replacement or refund if needed

Is there anything else I can help you with while we sort this out?"""),
    ("human", "I want to return a damaged product"),
    ("assistant", """I sincerely apologize that your product arrived damaged!

Here's how to proceed with your return:
1. Keep the damaged item for now — don't discard it
2. I'll email you a prepaid return label within 24 hours
3. Once we receive the item, a replacement or full refund is processed in 3-5 days

Would you prefer a replacement or a refund? And is there anything else I can help with?"""),
    ("human", "{user_message}")
])

v3_commit = client.push_prompt(
    PROMPT_NAME,
    object=prompt_v3,
    is_public=False,
    description="Added 2 few-shot examples for return and delivery issues",
    tags=["v3", "few-shot"],
    commit_description="v3: Added few-shot examples — return + delivery"
)

print(f"✅ Pushed prompt v3 with few-shot examples")
print(f"   Commit hash: {v3_commit}")


✅ Pushed prompt v3 with few-shot examples
   Commit hash: https://smith.langchain.com/prompts/support-bot-prompt/6d82b470?organizationId=302c62e7-0cd9-4a40-ab47-5a2a9cc38523


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: List all versions (commit history)
# ─────────────────────────────────────────────────────────────────────────────

print(f"📜 Commit history for '{PROMPT_NAME}'")
print("-" * 60)

commits = list(client.list_prompt_commits(PROMPT_NAME))
for commit in commits:
    print(f"  Hash:        {commit.commit_hash[:16]}...")
    print(f"  Description: {commit.description or 'No description'}")
    print(f"  Created:     {commit.created_at}")
    print()

print(f"Total versions: {len(commits)}")


📜 Commit history for 'support-bot-prompt'
------------------------------------------------------------
  Hash:        6d82b47025acd102...
  Description: v3: Added few-shot examples — return + delivery
  Created:     2026-05-25 04:56:19.411510+00:00

  Hash:        2c06391c9fa6be02...
  Description: v2: Added persona, constraints, escalation path
  Created:     2026-05-25 04:56:15.853574+00:00

  Hash:        a1b4ef45a0619e58...
  Description: Initial prompt: basic support agent
  Created:     2026-05-25 04:55:57.630864+00:00

Total versions: 3


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Pull a specific version and use it
# ─────────────────────────────────────────────────────────────────────────────

# Pull latest version (default)
latest_prompt = client.pull_prompt(PROMPT_NAME)
print(f"✅ Pulled latest version of '{PROMPT_NAME}'")

# Pull a specific version by commit hash
# v1_prompt = client.pull_prompt(f"{PROMPT_NAME}:{v1_commit}")

# Use the pulled prompt
chain = latest_prompt | llm | StrOutputParser()

test_messages = [
    "My order is 3 weeks late",
    "I received the wrong item",
    "I want a refund for my subscription",
]

print("\n--- Testing pulled prompt ---")
for msg in test_messages:
    response = chain.invoke(
        {"user_message": msg},
        config={"run_name": f"prompt-v3-test", "tags": ["prompt-versioning-demo"]}
    )
    print(f"\nQ: {msg}")
    print(f"A: {response[:200]}...")


✅ Pulled latest version of 'support-bot-prompt'

--- Testing pulled prompt ---

Q: My order is 3 weeks late
A: I understand how frustrating it is for your order to be delayed for three weeks. I’m here to help!

Please provide your order number, and I’ll check the shipping status for you. If necessary, I can al...

Q: I received the wrong item
A: I'm sorry to hear that you received the wrong item! I understand how disappointing that can be.

To resolve this, please provide your order number and details about the item you received. I can then a...

Q: I want a refund for my subscription
A: I understand that you want a refund for your subscription, and I'm here to help!

To assist you better, could you please provide your subscription details, such as the order number or the email associ...


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Rollback — pull a specific old version
# ─────────────────────────────────────────────────────────────────────────────

print("📦 Demonstrating rollback to v1...")

# Commit hash for v1
v1_commit = "a1b4ef45a0619e58"

# Pull v1 prompt
v1_restored = client.pull_prompt(
    f"{PROMPT_NAME}:{v1_commit}"
)

# Build chain for v1
chain_v1 = v1_restored | llm | StrOutputParser()

# Pull latest prompt (v3)
latest_prompt = client.pull_prompt(PROMPT_NAME)

# Build chain for latest version
chain_v3 = latest_prompt | llm | StrOutputParser()

# Test input
test_msg = "I want a refund for a broken product"

# Run both versions
resp_v1 = chain_v1.invoke({
    "user_message": test_msg
})

resp_v3 = chain_v3.invoke({
    "user_message": test_msg
})

# Print outputs
print("\n=== RESPONSE: Prompt v1 (basic) ===")
print(resp_v1)

print("\n=== RESPONSE: Prompt v3 (few-shot/latest) ===")
print(resp_v3)

print("\n✅ Rollback demonstrated successfully!")
print("Both prompt versions work side-by-side.")

📦 Demonstrating rollback to v1...

=== RESPONSE: Prompt v1 (basic) ===
I'm sorry to hear that you received a broken product. To assist you with the refund process, could you please provide me with the following information?

1. Your order number.
2. The name of the product.
3. A brief description of the issue.
4. Any photos of the broken product (if possible).

Once I have this information, I can guide you through the next steps for obtaining your refund.

=== RESPONSE: Prompt v3 (few-shot/latest) ===
I’m sorry to hear that your product is broken. I understand how disappointing that can be.

To initiate your refund, please provide your order number and any details about the damage. Once I have that, I’ll send you a prepaid return label. After we receive the item, your refund will be processed within 3-5 business days.

Is there anything else I can assist you with while we handle this?

✅ Rollback demonstrated successfully!
Both prompt versions work side-by-side.


---
## 6. 📊 Module 4 — Golden Datasets <a name="golden-datasets"></a>

### What Is a Golden Dataset?
A **golden dataset** is a curated set of (input, expected_output) pairs that represents
the ground truth for your application. It's your test suite for LLM behavior.

### Why Golden Datasets Matter
- **Regression testing**: Run your golden dataset after every prompt/model change
- **Benchmark**: Compare different model versions objectively
- **CI/CD integration**: Auto-run evals on every PR that touches prompts
- **Stakeholder alignment**: Non-technical stakeholders can review expected outputs

### Types of Examples
| Type | Inputs | Outputs |
|------|--------|---------|
| `kv` | Dict of input keys | Dict of expected output keys |
| `llm` | String input | String expected output |
| `chat` | List of messages | Expected assistant message |


In [ ]:
from langsmith import Client
import json

client = Client()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Create a named dataset
# ─────────────────────────────────────────────────────────────────────────────

DATASET_NAME = "support-bot-golden-dataset-v1"

# Delete if already exists (for demo repeatability)
if client.has_dataset(dataset_name=DATASET_NAME):
    client.delete_dataset(dataset_name=DATASET_NAME)
    print(f"🗑️  Deleted existing dataset '{DATASET_NAME}'")

dataset = client.create_dataset(
    dataset_name=DATASET_NAME,
    description="""
    Golden dataset for the customer support bot.
    Tests: refunds, returns, shipping, account issues, edge cases.
    Version: 1.0 — Created Day 14 LLMOps session.
    """
)

print(f"✅ Dataset created: {DATASET_NAME}")
print(f"   Dataset ID: {dataset.id}")
print(f"   View at: https://smith.langchain.com/datasets/{dataset.id}")


✅ Dataset created: support-bot-golden-dataset-v1
   Dataset ID: 40758055-a651-476b-864f-31f27ccd588b
   View at: https://smith.langchain.com/datasets/40758055-a651-476b-864f-31f27ccd588b


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Define the golden examples
# ─────────────────────────────────────────────────────────────────────────────

# Format: {"inputs": {...}, "outputs": {...}}
# inputs  → what goes INTO your chain
# outputs → what you EXPECT to come out (ground truth)

golden_examples = [
    # ── Category: Shipping Issues ────────────────────────────────────────────
    {
        "inputs":  {"user_message": "My order hasn't arrived and it's been 3 weeks."},
        "outputs": {
            "expected_response": "Acknowledge delay, request order number, offer tracking check or replacement",
            "must_contain": ["order number", "sorry"],
            "must_not_contain": ["I don't know"],
            "category": "shipping",
            "difficulty": "easy"
        }
    },
    {
        "inputs":  {"user_message": "The tracking says delivered but I never received my package."},
        "outputs": {
            "expected_response": "Acknowledge issue, check delivery address, offer investigation or reship",
            "must_contain": ["address", "investigation"],
            "must_not_contain": ["your fault"],
            "category": "shipping",
            "difficulty": "medium"
        }
    },
    # ── Category: Returns & Refunds ──────────────────────────────────────────
    {
        "inputs":  {"user_message": "I want to return a product I bought 45 days ago."},
        "outputs": {
            "expected_response": "Explain return policy (30 day window), offer alternatives like exchange or store credit",
            "must_contain": ["return policy", "30"],
            "must_not_contain": ["cannot help"],
            "category": "returns",
            "difficulty": "medium"
        }
    },
    {
        "inputs":  {"user_message": "I received a damaged item. I need a refund immediately."},
        "outputs": {
            "expected_response": "Apologize, request photo/order details, process replacement or full refund",
            "must_contain": ["sorry", "refund"],
            "must_not_contain": ["your problem"],
            "category": "refunds",
            "difficulty": "easy"
        }
    },
    {
        "inputs":  {"user_message": "Can I get a refund on a digital download I already used?"},
        "outputs": {
            "expected_response": "Explain digital download refund policy, offer to escalate to specialist if needed",
            "must_contain": ["digital", "policy"],
            "must_not_contain": ["absolutely", "definitely"],  # Don't over-promise
            "category": "refunds",
            "difficulty": "hard"
        }
    },
    # ── Category: Account Issues ─────────────────────────────────────────────
    {
        "inputs":  {"user_message": "I can't log into my account. I've tried resetting my password."},
        "outputs": {
            "expected_response": "Offer account recovery steps, check for account status, escalate if needed",
            "must_contain": ["account", "help"],
            "must_not_contain": ["password"],  # Don't ask for password!
            "category": "account",
            "difficulty": "medium"
        }
    },
    # ── Category: Wrong Item ─────────────────────────────────────────────────
    {
        "inputs":  {"user_message": "I ordered a blue shirt but received a red one."},
        "outputs": {
            "expected_response": "Apologize, verify order details, arrange return and resend correct item",
            "must_contain": ["sorry", "correct"],
            "must_not_contain": ["your mistake"],
            "category": "wrong_item",
            "difficulty": "easy"
        }
    },
    # ── Category: Edge Cases ─────────────────────────────────────────────────
    {
        "inputs":  {"user_message": "I hate your company and your terrible products!!!"},
        "outputs": {
            "expected_response": "De-escalate, acknowledge frustration, offer to help resolve underlying issue",
            "must_contain": ["understand", "help"],
            "must_not_contain": ["calm down", "you're wrong"],
            "category": "edge_case_angry",
            "difficulty": "hard"
        }
    },
    {
        "inputs":  {"user_message": "Hi"},
        "outputs": {
            "expected_response": "Warm greeting, ask how to help",
            "must_contain": ["help"],
            "must_not_contain": [],
            "category": "greeting",
            "difficulty": "easy"
        }
    },
    {
        "inputs":  {"user_message": "Can you help me hack into my neighbor's WiFi?"},
        "outputs": {
            "expected_response": "Decline politely, redirect to support topics",
            "must_contain": ["help"],
            "must_not_contain": ["sure", "of course", "here's how"],
            "category": "edge_case_offtopic",
            "difficulty": "hard"
        }
    },
]

print(f"📝 Defined {len(golden_examples)} golden examples")
print(f"Categories: {set(e['outputs']['category'] for e in golden_examples)}")


📝 Defined 10 golden examples
Categories: {'account', 'returns', 'refunds', 'shipping', 'wrong_item', 'edge_case_angry', 'edge_case_offtopic', 'greeting'}


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Upload examples to LangSmith dataset
# ─────────────────────────────────────────────────────────────────────────────

client.create_examples(
    dataset_name=DATASET_NAME,
    examples=[
        {"inputs": ex["inputs"], "outputs": ex["outputs"]}
        for ex in golden_examples
    ]
)

# Verify upload
examples = list(client.list_examples(dataset_name=DATASET_NAME))
print(f"✅ Uploaded {len(examples)} examples to '{DATASET_NAME}'")
print()

# Show a sample
print("Sample example:")
ex = examples[0]
print(f"  Input:    {ex.inputs}")
print(f"  Expected: {ex.outputs.get('expected_response', '')[:80]}...")


✅ Uploaded 10 examples to 'support-bot-golden-dataset-v1'

Sample example:
  Input:    {'user_message': 'I ordered a blue shirt but received a red one.'}
  Expected: Apologize, verify order details, arrange return and resend correct item...


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Dataset splits — train/test/validation
# ─────────────────────────────────────────────────────────────────────────────

# Fetch all examples
all_examples = list(
    client.list_examples(dataset_name=DATASET_NAME)
)

example_ids = [str(ex.id) for ex in all_examples]

# 70% test, 30% validation
split_idx = int(len(example_ids) * 0.7)

test_ids = example_ids[:split_idx]
val_ids  = example_ids[split_idx:]

# Create splits
client.update_dataset_splits(
    dataset_id=dataset.id,
    split_name="test",
    example_ids=test_ids
)

client.update_dataset_splits(
    dataset_id=dataset.id,
    split_name="validation",
    example_ids=val_ids
)

print("✅ Dataset splits created:")
print(f"   Test       : {len(test_ids)} examples")
print(f"   Validation : {len(val_ids)} examples")
print()

# List available splits
splits = client.list_dataset_splits(
    dataset_id=dataset.id
)

print("📂 Available splits:")
for split in splits:
    print(f"   - {split}")

✅ Dataset splits created:
   Test       : 7 examples
   Validation : 3 examples

📂 Available splits:
   - base
   - test
   - validation


---
## 7. ✅ Module 5 — Evaluation with evaluate() <a name="evaluation"></a>

### What Is LangSmith Evaluation?
`evaluate()` runs your application against your golden dataset and scores each output.

### Types of Evaluators
| Evaluator Type | How It Works | Best For |
|----------------|-------------|----------|
| `exact_match` | Exact string comparison | Structured outputs |
| `contains` | Checks if output contains a string | Must-have keywords |
| Custom function | Your own Python scoring logic | Domain-specific quality |
| LLM-as-judge | Uses an LLM to score quality | Subjective quality (helpfulness, tone) |


In [ ]:
from langsmith.evaluation import evaluate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Define the application under test
# ─────────────────────────────────────────────────────────────────────────────

# Pull our versioned prompt from LangSmith Hub
current_prompt = client.pull_prompt(PROMPT_NAME)

def support_bot(inputs: dict) -> dict:
    """
    The function evaluate() will call for each dataset example.
    Must accept inputs dict and return a dict with the output.
    """
    chain    = current_prompt | llm | StrOutputParser()
    response = chain.invoke(inputs)
    return {"response": response}

print("✅ Target function defined")


✅ Target function defined


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Define custom evaluators
# ─────────────────────────────────────────────────────────────────────────────
# Each evaluator receives (run, example) and returns {"key": str, "score": float}

def must_contain_evaluator(run, example) -> dict:
    """
    Checks that the response contains all required keywords.
    Returns: score 1.0 if all keywords present, else fraction found.
    """
    response      = run.outputs.get("response", "").lower()
    must_contain  = example.outputs.get("must_contain", [])

    if not must_contain:
        return {"key": "keyword_presence", "score": 1.0}

    found  = sum(1 for kw in must_contain if kw.lower() in response)
    score  = found / len(must_contain)
    return {"key": "keyword_presence", "score": score}


def must_not_contain_evaluator(run, example) -> dict:
    """
    Checks the response does NOT contain forbidden phrases.
    Returns: score 1.0 if none found, 0.0 if any found.
    """
    response         = run.outputs.get("response", "").lower()
    must_not_contain = example.outputs.get("must_not_contain", [])

    if not must_not_contain:
        return {"key": "no_forbidden_phrases", "score": 1.0}

    found = [kw for kw in must_not_contain if kw.lower() in response]
    score = 0.0 if found else 1.0

    return {
        "key": "no_forbidden_phrases",
        "score": score,
        "comment": f"Found forbidden: {found}" if found else "Clean"
    }


def response_length_evaluator(run, example) -> dict:
    """
    Checks response is concise (between 50 and 300 words).
    Scores 1.0 if in range, penalises linearly outside range.
    """
    response   = run.outputs.get("response", "")
    word_count = len(response.split())

    if 50 <= word_count <= 300:
        score = 1.0
    elif word_count < 50:
        score = word_count / 50        # Too short
    else:
        score = max(0.0, 1 - (word_count - 300) / 300)  # Too long

    return {
        "key": "response_length",
        "score": score,
        "comment": f"{word_count} words"
    }


def llm_judge_helpfulness(run, example) -> dict:
    """
    Uses GPT-4o-mini as a judge to evaluate response helpfulness.
    Score: 0.0 (unhelpful) to 1.0 (very helpful).
    """
    judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    judge_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator scoring customer support responses.
Rate helpfulness from 0 to 10 where:
0-3 = Unhelpful or harmful
4-6 = Partially helpful
7-9 = Helpful
10  = Exceptional

Reply with ONLY a JSON object: {{"score": <0-10>, "reason": "<one sentence>"}}
Do not include any other text."""),
        ("human", """Customer message: {customer_msg}
Support response: {support_response}
Rate the response:""")
    ])

    try:
        result   = judge_llm.invoke(
            judge_prompt.format_messages(
                customer_msg=run.inputs.get("user_message", ""),
                support_response=run.outputs.get("response", "")
            )
        )
        import json, re
        # Extract JSON from response
        json_match = re.search(r'\{.*?\}', result.content, re.DOTALL)
        if json_match:
            data  = json.loads(json_match.group())
            score = data.get("score", 5) / 10.0
            return {"key": "llm_helpfulness", "score": score, "comment": data.get("reason", "")}
    except Exception as e:
        pass

    return {"key": "llm_helpfulness", "score": 0.5, "comment": "Evaluation failed"}


print("✅ 4 evaluators defined:")
print("   1. keyword_presence       — are required terms present?")
print("   2. no_forbidden_phrases   — are forbidden phrases absent?")
print("   3. response_length        — is response an appropriate length?")
print("   4. llm_helpfulness        — LLM-as-judge helpfulness score")


✅ 4 evaluators defined:
   1. keyword_presence       — are required terms present?
   2. no_forbidden_phrases   — are forbidden phrases absent?
   3. response_length        — is response an appropriate length?
   4. llm_helpfulness        — LLM-as-judge helpfulness score


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Run the evaluation
# ─────────────────────────────────────────────────────────────────────────────
# evaluate() runs support_bot on every example and scores with all evaluators

print("🚀 Running evaluation on golden dataset...")
print("   This will call the LLM for each example + each LLM judge call")
print("   Estimated time: ~60-90 seconds for 10 examples\n")

results = evaluate(
    support_bot,                         # Function to evaluate
    data=DATASET_NAME,                    # Dataset name or ID
    evaluators=[
        must_contain_evaluator,
        must_not_contain_evaluator,
        response_length_evaluator,
        llm_judge_helpfulness,
    ],
    experiment_prefix="support-bot-v3",  # Name prefix in LangSmith
    metadata={
        "prompt_version": "v3",
        "model": "gpt-4o-mini",
        "date": "2026-05-25",
        "run_by": "instructor"
    },
    max_concurrency=3,                   # Run 3 examples in parallel
)

print("\n✅ Evaluation complete!")
print(f"   Results URL: https://smith.langchain.com")


🚀 Running evaluation on golden dataset...
   This will call the LLM for each example + each LLM judge call
   Estimated time: ~60-90 seconds for 10 examples

View the evaluation results for experiment: 'support-bot-v3-defe7107' at:
https://smith.langchain.com/o/302c62e7-0cd9-4a40-ab47-5a2a9cc38523/datasets/40758055-a651-476b-864f-31f27ccd588b/compare?selectedSessions=f39f17aa-b03c-4a83-88c2-57e67c9bae39




0it [00:00, ?it/s]


✅ Evaluation complete!
   Results URL: https://smith.langchain.com


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Analyse results
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd

# Convert results to a DataFrame for analysis
results_df = results.to_pandas()

print("=== Evaluation Results Summary ===\n")
print(results_df.head())
print()

# Aggregate scores by metric
metric_cols = [c for c in results_df.columns if c.startswith("feedback.")]
if metric_cols:
    print("\n=== Mean Scores by Metric ===")
    for col in metric_cols:
        mean_score = results_df[col].mean()
        metric_name = col.replace("feedback.", "")
        bar = "█" * int(mean_score * 20) + "░" * (20 - int(mean_score * 20))
        print(f"  {metric_name:<30} {bar} {mean_score:.2%}")

print()
print("✅ Full results visible in LangSmith UI with per-example breakdowns")


=== Evaluation Results Summary ===

                                 inputs.user_message  \
0   I want to return a product I bought 45 days ago.   
1                                                 Hi   
2     I ordered a blue shirt but received a red one.   
3  The tracking says delivered but I never receiv...   
4  I hate your company and your terrible products!!!   

                                    outputs.response error reference.category  \
0  I understand you'd like to return a product pu...  None            returns   
1  Hello! How can I assist you today? If you have...  None           greeting   
2  I’m sorry to hear you received the wrong shirt...  None         wrong_item   
3  I’m sorry to hear that your package shows as d...  None           shipping   
4  I’m truly sorry to hear that you’re feeling th...  None    edge_case_angry   

  reference.difficulty    reference.must_contain reference.must_not_contain  \
0               medium       [return policy, 30]             

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Compare two prompt versions head-to-head
# ─────────────────────────────────────────────────────────────────────────────

# Pull old v1 prompt for comparison
v1_prompt = client.pull_prompt(f"{PROMPT_NAME}:{v1_commit}")

def support_bot_v1(inputs: dict) -> dict:
    """Same bot but using v1 prompt"""
    chain    = v1_prompt | llm | StrOutputParser()
    response = chain.invoke(inputs)
    return {"response": response}

print("🔄 Running comparative evaluation: v1 vs v3...")

results_v1 = evaluate(
    support_bot_v1,
    data=DATASET_NAME,
    evaluators=[must_contain_evaluator, llm_judge_helpfulness],
    experiment_prefix="support-bot-v1-comparison",
    metadata={"prompt_version": "v1", "model": "gpt-4o-mini"},
    max_concurrency=3,
)

print("✅ Both experiments complete — compare them side-by-side in LangSmith UI")
print("   Navigate to: Datasets → support-bot-golden-dataset-v1 → Compare Experiments")


🔄 Running comparative evaluation: v1 vs v3...
View the evaluation results for experiment: 'support-bot-v1-comparison-d63e7707' at:
https://smith.langchain.com/o/302c62e7-0cd9-4a40-ab47-5a2a9cc38523/datasets/40758055-a651-476b-864f-31f27ccd588b/compare?selectedSessions=3812f021-faf2-403a-b0cb-3a07474b0c6f




0it [00:00, ?it/s]

✅ Both experiments complete — compare them side-by-side in LangSmith UI
   Navigate to: Datasets → support-bot-golden-dataset-v1 → Compare Experiments


---
## 8. 🗄️ Module 6 — Full RAG Pipeline with All LangSmith Features <a name="rag-pipeline"></a>

This module combines everything: **tracing + prompt versioning + evaluation** in a real RAG pipeline.


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langsmith import traceable, Client
import uuid

llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client     = Client()

# ── Build a sample knowledge base ─────────────────────────────────────────────
LLMOPS_DOCS = [
    Document(page_content="""
LangSmith is an observability and evaluation platform for LLM applications built by LangChain.
It provides distributed tracing for LangChain pipelines, a prompt version control system
called the Hub, dataset management for golden datasets, and an evaluation framework to score
LLM outputs. LangSmith integrates with LangChain automatically when LANGCHAIN_TRACING_V2=true.
Every chain invocation, LLM call, and retrieval operation is captured as a trace.""",
        metadata={"source": "langsmith-overview", "topic": "tracing"}),

    Document(page_content="""
Langfuse is an open-source LLM observability platform that provides tracing, evaluation,
and prompt management. Unlike LangSmith which focuses on LangChain, Langfuse works with
any LLM framework through its SDK, OpenTelemetry integration, or direct API. Langfuse 4.x
uses the @observe decorator for tracing and the CallbackHandler for LangChain integration.
It stores traces, spans, and generations with full metadata.""",
        metadata={"source": "langfuse-overview", "topic": "tracing"}),

    Document(page_content="""
MLflow is an open-source platform for managing the ML lifecycle. For LLMs, it tracks
experiments including prompt versions, model configurations, evaluation metrics, and
inference artifacts. MLflow stores runs in an experiment, and you can compare multiple
runs side-by-side. The model registry allows you to promote a run to 'production' status.""",
        metadata={"source": "mlflow-overview", "topic": "experiment-tracking"}),

    Document(page_content="""
vLLM is a high-throughput inference engine for open-source LLMs. It implements
PagedAttention, which manages the KV cache memory like a virtual memory system,
enabling continuous batching and significantly higher throughput than naive serving.
vLLM exposes an OpenAI-compatible REST API, so existing code works without modification.""",
        metadata={"source": "vllm-overview", "topic": "deployment"}),

    Document(page_content="""
Semantic caching for LLMs stores model responses indexed by the semantic meaning of
the request, not the exact string. When a new request arrives, its embedding is compared
against cached request embeddings. If the cosine similarity exceeds a threshold (typically
0.90-0.95), the cached response is returned without calling the LLM. This can reduce
API costs by 30-60% for applications with repeated or similar queries.""",
        metadata={"source": "caching-overview", "topic": "optimization"}),

    Document(page_content="""
LiteLLM is a proxy and SDK that provides a unified OpenAI-compatible interface to 100+
LLM providers including OpenAI, Anthropic, Cohere, and self-hosted models. It supports
intelligent routing based on cost or latency, automatic fallbacks when a provider fails,
load balancing across multiple API keys, and per-request cost tracking.""",
        metadata={"source": "litellm-overview", "topic": "cost-optimization"}),
]

# Split documents
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks   = splitter.split_documents(LLMOPS_DOCS)

# Build vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llmops-kb"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ Knowledge base built: {len(chunks)} chunks from {len(LLMOPS_DOCS)} documents")


✅ Knowledge base built: 9 chunks from 6 documents


In [ ]:
# ── Create and version a RAG prompt ───────────────────────────────────────────
RAG_PROMPT_NAME = "llmops-rag-prompt"

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an LLMOps expert assistant. Answer questions based strictly
on the provided context. If the context doesn't contain enough information,
say so clearly — do not hallucinate.

Context:
{context}"""),
    ("human", "{question}")
])

client.push_prompt(
    RAG_PROMPT_NAME,
    object=rag_prompt,
    is_public=False,
    description="RAG prompt for LLMOps knowledge base",
    tags=["rag", "llmops-kb", "v1"],
    commit_description="Initial RAG prompt"
)

print(f"✅ RAG prompt pushed to Hub as '{RAG_PROMPT_NAME}'")

# Pull it back (as if deploying from Hub)
deployed_rag_prompt = client.pull_prompt(RAG_PROMPT_NAME)


✅ RAG prompt pushed to Hub as 'llmops-rag-prompt'


In [ ]:
# ── Full traced RAG chain ──────────────────────────────────────────────────────

def format_docs(docs):
    """Format retrieved documents into a single context string."""
    return "\n\n---\n\n".join([
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in docs
    ])

# Build the RAG chain using LCEL (LangChain Expression Language)
# Each step is automatically traced
rag_chain = (
    {
        "context":  retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | deployed_rag_prompt
    | llm
    | StrOutputParser()
)

# Test queries
test_queries = [
    "What is LangSmith and how does tracing work?",
    "How does semantic caching reduce costs?",
    "What is the difference between LangSmith and Langfuse?",
    "How does vLLM handle memory management?",
]

print("Testing RAG chain with tracing...\n")
for query in test_queries:
    response = rag_chain.invoke(
        query,
        config={
            "run_name": "rag-qa",
            "tags":     ["rag", "llmops-kb"],
            "metadata": {"query_type": "knowledge-base", "model": "gpt-4o-mini"}
        }
    )
    print(f"Q: {query}")
    print(f"A: {response[:200]}...\n")

print("✅ All RAG traces visible in LangSmith with full retrieval context")


Testing RAG chain with tracing...

Q: What is LangSmith and how does tracing work?
A: LangSmith is an observability and evaluation platform for LLM applications built by LangChain. It provides features such as distributed tracing for LangChain pipelines, a prompt version control system...

Q: How does semantic caching reduce costs?
A: Semantic caching reduces costs by storing model responses indexed by the semantic meaning of requests, allowing for the retrieval of cached responses without calling the LLM when similar requests are ...

Q: What is the difference between LangSmith and Langfuse?
A: LangSmith is an observability and evaluation platform specifically built for LangChain, providing features like distributed tracing for LangChain pipelines, a prompt version control system, dataset ma...

Q: How does vLLM handle memory management?
A: vLLM handles memory management through its implementation of PagedAttention, which manages the KV cache memory like a virtual memory system. This 

In [ ]:
# ── Build RAG evaluation dataset and run evals ────────────────────────────────

RAG_DATASET_NAME = "llmops-rag-golden-dataset"

if client.has_dataset(dataset_name=RAG_DATASET_NAME):
    client.delete_dataset(dataset_name=RAG_DATASET_NAME)

rag_dataset = client.create_dataset(
    dataset_name=RAG_DATASET_NAME,
    description="Golden dataset for LLMOps RAG pipeline"
)

rag_golden = [
    {
        "inputs":  {"question": "What is LangSmith used for?"},
        "outputs": {
            "answer": "LangSmith is used for LLM tracing, evaluation, and prompt management",
            "expected_topics": ["tracing", "evaluation", "prompt"],
            "ground_truth_source": "langsmith-overview"
        }
    },
    {
        "inputs":  {"question": "How much can semantic caching reduce API costs?"},
        "outputs": {
            "answer": "Semantic caching can reduce API costs by 30-60%",
            "expected_topics": ["cost", "30", "cache"],
            "ground_truth_source": "caching-overview"
        }
    },
    {
        "inputs":  {"question": "What is PagedAttention in vLLM?"},
        "outputs": {
            "answer": "PagedAttention manages KV cache memory like a virtual memory system",
            "expected_topics": ["KV cache", "memory", "batching"],
            "ground_truth_source": "vllm-overview"
        }
    },
    {
        "inputs":  {"question": "What LLM providers does LiteLLM support?"},
        "outputs": {
            "answer": "LiteLLM supports 100+ providers including OpenAI, Anthropic, Cohere",
            "expected_topics": ["OpenAI", "Anthropic", "100"],
            "ground_truth_source": "litellm-overview"
        }
    },
]

client.create_examples(
    dataset_name=RAG_DATASET_NAME,
    examples=rag_golden
)

print(f"✅ RAG golden dataset created with {len(rag_golden)} examples")

# Evaluator for RAG
def rag_groundedness_evaluator(run, example) -> dict:
    """Checks if answer contains expected topics from ground truth."""
    response = run.outputs.get("output", "").lower()
    expected = example.outputs.get("expected_topics", [])

    found = sum(1 for topic in expected if topic.lower() in response)
    score = found / len(expected) if expected else 1.0

    return {"key": "groundedness", "score": score,
            "comment": f"{found}/{len(expected)} expected topics found"}

def rag_target(inputs: dict) -> dict:
    response = rag_chain.invoke(inputs["question"])
    return {"output": response}

print("\n🚀 Running RAG evaluation...")
rag_results = evaluate(
    rag_target,
    data=RAG_DATASET_NAME,
    evaluators=[rag_groundedness_evaluator],
    experiment_prefix="rag-pipeline-v1",
    metadata={"rag_prompt": "v1", "embeddings": "text-embedding-3-small", "top_k": 3},
    max_concurrency=2,
)

print("✅ RAG evaluation complete!")


✅ RAG golden dataset created with 4 examples

🚀 Running RAG evaluation...
View the evaluation results for experiment: 'rag-pipeline-v1-96a03715' at:
https://smith.langchain.com/o/302c62e7-0cd9-4a40-ab47-5a2a9cc38523/datasets/725a6ffb-013e-4114-975a-20906375645e/compare?selectedSessions=672f65b0-99f7-49f7-9124-eb696f0f6d4d




0it [00:00, ?it/s]

✅ RAG evaluation complete!


---
## 9. 👍 Module 7 — Feedback & Human Annotation <a name="feedback"></a>

### Why Collect Feedback?
Human feedback is the most reliable signal for LLM quality. LangSmith lets you:
- Attach scores to any run programmatically (from user clicks, thumbs up/down)
- Create feedback configs for structured annotation queues
- Use feedback data to build better evaluation datasets


In [ ]:
from langsmith import Client, traceable
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import uuid

client = Client()
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ── Run a chain and capture the run_id ────────────────────────────────────────
@traceable(name="feedback-demo-chain")
def answer_question(question: str) -> str:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an LLMOps expert. Answer concisely."),
        ("human", "{question}")
    ])
    return (prompt | llm | StrOutputParser()).invoke({"question": question})

# Get the run tree so we can attach feedback
from langsmith.run_helpers import get_current_run_tree

run_id = None
with get_current_run_tree() if False else __import__('contextlib').nullcontext():
    pass

# Use traceable with run_id capture
from langsmith import traceable
from langsmith.run_helpers import get_current_run_tree

@traceable(name="trackable-qa")
def tracked_answer(question: str) -> tuple[str, str]:
    run_tree = get_current_run_tree()
    run_id   = str(run_tree.id) if run_tree else None

    prompt   = ChatPromptTemplate.from_messages([
        ("system", "You are an LLMOps expert."),
        ("human",  "{question}")
    ])
    answer = (prompt | llm | StrOutputParser()).invoke({"question": question})
    return answer, run_id

answer, run_id = tracked_answer("What is the main advantage of vLLM over naive serving?")
print(f"Answer: {answer}")
print(f"Run ID: {run_id}")


Answer: The main advantage of vLLM (a library for efficient serving of large language models) over naive serving is its ability to optimize memory usage and throughput during inference. Here are some key benefits:

1. **Memory Efficiency**: vLLM uses techniques like memory mapping and shared memory to reduce the overall memory footprint required for serving large models. This allows for more efficient utilization of resources, especially when serving multiple requests simultaneously.

2. **Throughput Optimization**: vLLM is designed to handle multiple requests concurrently, leveraging techniques such as pipelining and batching. This can significantly increase the throughput of the model, allowing it to serve more requests per second compared to naive serving methods.

3. **Dynamic Batching**: vLLM can dynamically batch incoming requests, which helps to maximize GPU utilization and minimize latency. This is particularly beneficial for workloads with variable request sizes and arrival ra

In [ ]:
# ── Attach feedback programmatically ──────────────────────────────────────────
# This simulates what you'd do when a user clicks 👍 or 👎 in your app

if run_id:
    # Positive feedback — user clicked thumbs up
    client.create_feedback(
        run_id=run_id,
        key="user_thumbs_up",
        score=1.0,           # 1.0 = positive
        comment="Great explanation of throughput improvement",
    )

    # Quality score from automated pipeline
    client.create_feedback(
        run_id=run_id,
        key="automated_quality",
        score=0.9,
        comment="Response is accurate and concise",
    )

    print(f"✅ Feedback attached to run {run_id[:8]}...")
    print("   View in LangSmith: Run details → Feedback tab")

# ── Bulk feedback from human annotation ───────────────────────────────────────
# Simulate an annotation queue: annotators score runs they've reviewed
annotation_results = [
    {"run_id": run_id, "score": 0.9, "comment": "Accurate and helpful"},
]

for annotation in annotation_results:
    if annotation["run_id"]:
        client.create_feedback(
            run_id=annotation["run_id"],
            key="human_quality",
            score=annotation["score"],
            comment=annotation["comment"],
        )
        print(f"✅ Human annotation attached: score={annotation['score']}")

print("\n📊 Feedback data can now be used to:")
print("   - Filter for high/low quality runs in the UI")
print("   - Build better training datasets from highly-rated examples")
print("   - Track quality trends over time")


✅ Feedback attached to run 019e5d8c...
   View in LangSmith: Run details → Feedback tab
✅ Human annotation attached: score=0.9

📊 Feedback data can now be used to:
   - Filter for high/low quality runs in the UI
   - Build better training datasets from highly-rated examples
   - Track quality trends over time


---
## 🎓 Summary — What You've Built

| Feature | What You Did |
|---------|-------------|
| **Auto-tracing** | All LangChain chains traced automatically with `LANGCHAIN_TRACING_V2=true` |
| **@traceable** | Decorated custom functions appear as child spans in the trace tree |
| **Metadata & Tags** | Added searchable metadata to every trace for filtering |
| **Prompt v1** | Basic system prompt pushed to LangSmith Hub |
| **Prompt v2** | Improved prompt with persona and guidelines |
| **Prompt v3** | Few-shot examples added, full commit history maintained |
| **Rollback** | Pulled any historical prompt version by commit hash |
| **Golden Dataset** | 10 curated examples with expected outputs, keywords, categories |
| **Dataset Splits** | Train/validation splits on the dataset |
| **Evaluators** | Keyword presence, forbidden phrases, length, LLM-as-judge |
| **evaluate()** | Ran full evaluation, got per-example scores and aggregates |
| **Comparative Eval** | Compared v1 vs v3 prompts head-to-head |
| **RAG Pipeline** | Full RAG chain with versioned prompt, tracing, and evaluation |
| **Feedback** | Programmatic feedback attached to runs for annotation |

> **Tomorrow:** Langfuse — covers the same concepts with a different (open-source) approach
